In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. **Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs**
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Multi-Agent Deitel Book Concierge with `FileSearchTool` and `WebSearchTool`
* Both are OpenAI-hosted and work only with OpenAI models
* `FileSearchTool` searches a vector store hosted in your OpenAI developer account
    * Searching private vector stores requires custom `@function_tool`s
---

## Retrieval-Augmented Generation (RAG)
* Helps an agent answer from indexed files
* Augments LLM's trained knowledge with additional info
    * e.g., company-specific knowledge that LLM would not know
* **Vector stores** save **embeddings**
    * Mathematical representations text's meaning
    * Supports **semantic search**
        * Matches meanings, not exact words
        * E.g., search for "car repair" might find content on "fixing an automobile"
        * Quickly find semantically similar content
        * \[Side note: If you like word games, try Semantle\]
* **`FileSearchTool`**
    * Performs semantic search to retrieve relevant passages during agent run
* **Hybrid RAG**
    * **RAG + web search** so agent can also search online for info not in vector store
---


## Multi-Agent Pattern: Handoffs
* SDK mechanism for one agent to delegate control to another
* **Triage agent**
    * Uses LLM reasoning to route requests to specialist agents 
    * **Transfers control** — specialist owns next reply after receiving handoff
* Use when each specialists has different instructions, tools, or domains
    * Routing workflows, concierge, support, tutoring, ...
    * This example is effectively a customer-service agent

---

## `RECOMMENDED_PROMPT_PREFIX` Constant
* Provided by SDK
* Prepend to agents that participate in handoffs
    * agents that initiate handoffs
    * agents that can receive handoffs
* Tells agent it's part of a multi-agent system and that transfers between agents should stay invisible to the user

In [ ]:
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
print(RECOMMENDED_PROMPT_PREFIX)

---
## Indexed Files in This Example
| Agent | Vector Store Name | Book |
|---|---|---|
| `JavaConcierge` | `jhtp12` | *Java How to Program, 12/e* — Preface, TOC, Index |
| `PythonConcierge` | `IntroToPython1` | *Intro to Python for Computer Science & Data Science* — Preface, TOC, Index |
---

## Uploading Files to an OpenAI Vector Store Via the Web Interface

The easiest way to create a vector store and add files is through the OpenAI Platform dashboard:

1. Go to **[platform.openai.com/storage](https://platform.openai.com/storage)**
2. In the Storage page, select the **"Vector stores"** tab, then click **"+ Create"**
3. Give the vector store a name and click the checkmark icon
4. At the bottom of the screen, click **"+ Add files"**
5. Upload your file — OpenAI chunks, embeds, and indexes them automatically
    * **Supported file types:** PDF, DOCX, TXT, MD, HTML, JSON, PPTX, and more
    * **Processing time to convert file to vector store** — seconds to minutes depending on file size
6. Copy the **vector store ID** (format: `vs_...`) — you'll use it in your code

## Vector Store IDs

In [ ]:
JHTP_VS_ID = 'vs_6a260a96a6148191bba23f657fcf90ea'  # Java How to Program 12/e
PYTHON_VS_ID = 'vs_6a260ba3bcdc8191952f212e5555665a'  # Intro to Python

In [ ]:
from openai import OpenAI
client = OpenAI()

for store_id in [JHTP_VS_ID, PYTHON_VS_ID]:
    store = client.vector_stores.retrieve(store_id)
    print(f'{store.name}\nID={store_id}\n{store.status=}\n{store.usage_bytes=:,d}\n') 

---
## Create the Concierge Agents

### Imports

In [ ]:
import os
from agents import Agent, Runner, handoff, trace
from agents.extensions.handoff_prompt import (
    prompt_with_handoff_instructions
)
from agents.tool import FileSearchTool, WebSearchTool
from source.agent_loop import run_conversation

--- 
### Concierge Agent Instructions

In [ ]:
JAVA_INSTRUCTIONS = prompt_with_handoff_instructions("""
You are the Java book concierge for the textbook 
Java How to Program, 12/e by Deitel.

Scope:
- Answer only questions about this Java book, its table of contents, its
  index, its preface, Java topic coverage, Java chapter/section location,
  and Java study paths based on this book.
- If the request is unrelated to Java or this book, do not answer it. Say:
  "This concierge is limited to Java How to Program, 12/e. Please ask
  a question about that book or its Java topics."

Use the FileSearchTool first for book questions. Do not claim a topic is
covered unless file search supports it.

Every substantive answer MUST begin with this format:
**Book:** Java How to Program, 12/e
**Chapter/section:** <chapter/section if found; otherwise say 
not found in the indexed preface/TOC/index>

Then answer concisely in plain language.
Do not provide citations to the original documents.
""")


In [ ]:
PYTHON_INSTRUCTIONS = prompt_with_handoff_instructions("""
You are the Python book concierge for the textbook Intro to 
Python for Computer Science and Data Science by Deitel.

Scope:
- Answer only questions about this Python book, its table of contents, its
  index, its preface, Python/data-science topic coverage, chapter/section
  location, and study paths based on this book.
- If the request is unrelated to Python, data science or this book, do not
  answer it. Say: "This concierge is limited to Intro to Python for
  Computer Science and Data Science. Please ask a question about that book
  or its Python/data-science topics."

Use the FileSearchTool first for book questions. Do not claim a topic is
covered unless file search supports it.

Every substantive answer MUST begin with this format:
**Book:** Intro to Python for Computer Science and Data Science
**Chapter/section:** <chapter/section if found; otherwise say not 
found in the indexed preface/TOC/index>

Then answer concisely in plain language.
Do not provide citations to the original documents.
""")

--- 
### Web Search Instructions

In [ ]:
WEB_SUFFIX = """

You also have access to the WebSearchTool. Use it only to supplement an
in-scope book answers with:
- Current official documentation, tutorials, or API references
- Recent language or library updates not covered in the book edition

Do not use web search to answer unrelated or out-of-scope requests.
When you include web information, put it under a final heading:
**External resources:**
Include URLs to the relevant online resources.
"""

### Creating the `Agent` Objects
* Java and Python concierge agents use hybrid RAG
    * `FileSearchTool` — for indexed book content
    * `WebSearchTool` — only to supplement in-scope book answers with current documentation
* `GeneralSearchAgent` is limited to closely related programming/software questions


In [ ]:
java_agent = Agent(
    name='JavaConcierge',
    model='gpt-5.5',
    instructions=JAVA_INSTRUCTIONS + WEB_SUFFIX,
    tools=[
        FileSearchTool(
            vector_store_ids=[JHTP_VS_ID], # could be multiple vector stores
            max_num_results=10 # number of document chunks to return (1-50, 20 default)
        ),
        WebSearchTool()
    ]
)

python_agent = Agent(
    name='PythonConcierge',
    model='gpt-5.5',
    instructions=PYTHON_INSTRUCTIONS + WEB_SUFFIX,
    tools=[
        FileSearchTool(
            vector_store_ids=[PYTHON_VS_ID], 
            max_num_results=10
        ),
        WebSearchTool()
    ]
)

print('Specialist agents ready:', 
      java_agent.name, python_agent.name, sep='\n')

In [ ]:
GENERAL_SEARCH_INSTRUCTIONS = prompt_with_handoff_instructions("""
You are a programming-focused web-search specialist for the Deitel book
concierge demo.

Scope:
- Answer only programming, software-development, Java, Python, data-science,
  or official-documentation questions that are not specific to one indexed
  Deitel book.
- If the request is unrelated to programming, software development, Java,
  Python or data science, do not answer it. Say: "This demo is limited to
  Deitel Java/Python book questions and closely related programming topics."

Use WebSearchTool and prefer official documentation when possible.
Include URLs for the sources you use.
""")

general_search_agent = Agent(
    name='GeneralSearchAgent',
    model='gpt-5.5',
    instructions=GENERAL_SEARCH_INSTRUCTIONS,
    tools=[
        WebSearchTool()
    ]
)

print('Specialist agent ready:', general_search_agent.name)

## Create the Triage Agent
* Only job is determining which specialist agent should handle request
* Uses `handoff()` to specify agents that can receive a handoff
* Asks for clarification when user input is ambiguous
* Refuses unrelated requests instead of routing them to a general assistant


In [ ]:
TRIAGE_INSTRUCTIONS = prompt_with_handoff_instructions("""
You are a routing agent for a Deitel programming book concierge demo.
Available specialists:
- JavaConcierge — Java How to Program, 12/e
- PythonConcierge — Intro to Python for Computer Science and Data Science
- GeneralSearchAgent — programming/software web search for closely related
  topics that are not specific to one indexed Deitel book

Routing rules:
- Route Java book questions, Java topic-coverage questions, Java chapter/
  section questions, and Java study-path questions to JavaConcierge.
- Route Python book questions, Python/data-science topic-coverage questions,
  chapter/section questions, and study-path questions to PythonConcierge.
- If the question could apply to both books, ask exactly one clarifying
  question: "Do you mean the Java book or the Python book?" Do not route yet.
- Route closely related programming/software questions that are not specific
  to either book to GeneralSearchAgent.
- If the request is unrelated to programming, software development, Java,
  Python, data science or these Deitel books, do not hand off. Reply briefly:
  "This demo is limited to Deitel Java/Python book questions and closely
  related programming topics."

Do not answer Java or Python book questions yourself. Hand off to the
appropriate specialist.
""")

In [ ]:
triage_agent = Agent(
    name='BookConciergeRouter',
    model='gpt-5.5',
    instructions=TRIAGE_INSTRUCTIONS,
    handoffs=[
        handoff(java_agent),
        handoff(python_agent),
        handoff(general_search_agent)
    ]
)

print('Triage agent ready:', triage_agent.name)

## Visualizing the Agent Workflow

In [ ]:
from agents.extensions.visualization import draw_graph

In [ ]:
draw_graph(triage_agent, filename="agent_workflow.png")

## Run the Handoff-Based Concierge
* We'll use `run_conversation()` to launch the triage agent
* `Runner` manages handoffs from the triage agent to the specialists transparently

In [ ]:
prompt = """
Book Concierge ready! 

Ask about the textbooks Java How to Program 12/e or 
Intro to Python for Computer Science and Data Science. 
There is no need to name the book explicitly. 

Type "exit" to quit.
"""

with trace('02-05-03-book-concierge-file-search-handoffs'):
    await run_conversation(prompt, triage_agent)

### Sample prompts
* Where can I learn about functional programming?
    * Triage agent should ask which book, as there's no prior conversation state
* Where can I learn about Java inheritance?
* What about in Python?
* Suggest a data science study path.

---
## Post-Class Reference: Vector Store Operations

* Create vector stores in the dashboard or with `client.vector_stores.create(...)`
* Upload files with `client.vector_stores.file_batches.upload_and_poll(...)`
* Check actual storage with `client.vector_stores.retrieve(id).usage_bytes`
* Delete unused vector stores with `client.vector_stores.delete(id)`
* Delete uploaded files separately with `client.files.delete(file_id)`
* Use expiration policies for temporary or demo vector stores
* Pricing and storage details can change — check the current docs

**References:**
* [OpenAI Retrieval Guide](https://developers.openai.com/api/docs/guides/retrieval#vector-stores)
* [OpenAI File Search Guide](https://developers.openai.com/api/docs/guides/tools-file-search)

---


&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. 